# Sparse and dense text representations

TF-IDF and frozen MiniLM sentence vectors are compared using the same held-out partition. Both classifiers are selected using training folds only.

In [ ]:
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

categories = ['sci.space', 'rec.sport.baseball']
data = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))
train_ids, test_ids = train_test_split(
    np.arange(len(data.target)), test_size=0.2, random_state=42, stratify=data.target)
texts = np.asarray(data.data, dtype=object)
y = np.asarray(data.target)
cv = StratifiedKFold(3, shuffle=True, random_state=42)
baseline = GridSearchCV(
    Pipeline([('features', TfidfVectorizer(min_df=2)),
              ('classifier', LogisticRegression(max_iter=1000))]),
    {'classifier__C': [0.1, 1, 10]}, scoring='f1_macro', cv=cv)
baseline.fit(texts[train_ids].tolist(), y[train_ids])
predictions = baseline.predict(texts[test_ids].tolist())
print('TF-IDF held-out macro-F1:', round(f1_score(y[test_ids], predictions, average='macro'), 3))

## Frozen sentence embeddings

The encoder is used for feature extraction. The logistic-regression classifier is tuned separately; this experiment does not fine-tune the Transformer.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = encoder.encode(texts.tolist(), batch_size=32, show_progress_bar=True,
                            normalize_embeddings=True)
head = GridSearchCV(LogisticRegression(max_iter=1000), {'C': [0.1, 1, 10]},
                    scoring='f1_macro', cv=cv)
head.fit(embeddings[train_ids], y[train_ids])
predictions = head.predict(embeddings[test_ids])
print('Frozen encoder held-out macro-F1:',
      round(f1_score(y[test_ids], predictions, average='macro'), 3))